# PredMarket Arb — Data Exploration

Quick visual inspection of downloaded OHLCV datasets

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import duckdb
from pathlib import Path

plt.style.use('dark_background')
sns.set_palette("husl")

Matplotlib is building the font cache; this may take a moment.


In [2]:
ASSETS = ["BTCUSDT","ETHUSDT","SOLUSDT","XRPUSDT","DOGEUSDT","BNBUSDT","HYPEUSDT"]

summary = []
for asset in ASSETS:
    path = Path(f"data/raw/{asset}_1min.parquet")
    if not path.exists():
        print(f"⚠️  {asset} not found — run download_datasets.py first")
        continue
    df = pd.read_parquet(path, columns=["timestamp","close","volume"])
    summary.append({
        "Asset":    asset.replace("USDT",""),
        "Rows":     f"{len(df):,}",
        "Start":    df.timestamp.min().strftime("%Y-%m-%d"),
        "End":      df.timestamp.max().strftime("%Y-%m-%d"),
        "Size MB":  f"{path.stat().st_size / 1e6:.0f}",
        "Nulls":    df.isnull().sum().sum()
    })
pd.DataFrame(summary).set_index("Asset")

⚠️  BTCUSDT not found — run download_datasets.py first
⚠️  ETHUSDT not found — run download_datasets.py first
⚠️  SOLUSDT not found — run download_datasets.py first
⚠️  XRPUSDT not found — run download_datasets.py first
⚠️  DOGEUSDT not found — run download_datasets.py first
⚠️  BNBUSDT not found — run download_datasets.py first
⚠️  HYPEUSDT not found — run download_datasets.py first


KeyError: "None of ['Asset'] are in the columns"

## Price History — All Assets

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()

for i, asset in enumerate(ASSETS[:6]):
    path = Path(f"data/raw/{asset}_1min.parquet")
    if not path.exists(): continue
    
    # Load daily close (faster than 1min for plotting)
    df = pd.read_parquet(path, columns=["timestamp","close"])
    daily = df.set_index("timestamp")["close"].resample("1D").last().dropna()
    normalized = (daily / daily.iloc[0]) * 100
    
    axes[i].plot(normalized.index, normalized.values, linewidth=1)
    axes[i].set_title(asset.replace("USDT",""), fontsize=14, fontweight='bold')
    axes[i].set_ylabel("Indexed (start=100)")
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[i].grid(alpha=0.3)

plt.suptitle("Price History — Normalized to 100 at Start", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Gap Analysis — Are there holes in the data?

In [ ]:
for asset in ASSETS:
    path = Path(f"data/raw/{asset}_1min.parquet")
    if not path.exists(): continue
    
    df = pd.read_parquet(path, columns=["timestamp"])
    gaps = df["timestamp"].diff()
    
    gap_5min  = (gaps > pd.Timedelta("5min")).sum()
    gap_1h    = (gaps > pd.Timedelta("1h")).sum()
    gap_1day  = (gaps > pd.Timedelta("1D")).sum()
    
    status = "✅" if gap_5min < 100 else "⚠️ "
    print(f"{status} {asset:<12} gaps>5min: {gap_5min:>5} | "
          f"gaps>1h: {gap_1h:>4} | gaps>1day: {gap_1day:>3}")

## Volume Distribution — Detecting Dead Candles

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, asset in enumerate(ASSETS[:6]):
    path = Path(f"data/raw/{asset}_1min.parquet")
    if not path.exists(): continue
    
    df = pd.read_parquet(path, columns=["volume"])
    vol = df["volume"][df["volume"] > 0]
    
    axes[i].hist(np.log1p(vol), bins=100, alpha=0.7, edgecolor='none')
    axes[i].set_title(asset.replace("USDT",""))
    axes[i].set_xlabel("log(volume + 1)")
    axes[i].set_ylabel("Count")
    axes[i].grid(alpha=0.3)

plt.suptitle("Volume Distribution (log scale) — look for anomalies", fontsize=14)
plt.tight_layout()
plt.show()

## Quick DuckDB Query — Top 10 Most Volatile Days

In [ ]:
result = duckdb.query("""
    SELECT 
        strftime(timestamp, '%Y-%m-%d')     AS day,
        round(max(high) - min(low), 2)      AS range_usd,
        round(max(high)/min(low) - 1, 4)    AS range_pct,
        round(sum(volume), 0)               AS total_volume
    FROM 'data/raw/BTCUSDT_1min.parquet'
    GROUP BY day
    ORDER BY range_pct DESC
    LIMIT 10
""").df()

print("🔥 Top 10 most volatile days for BTC:")
result

## ✅ Data looks good — next step: feature_engineering.py
Next notebook: 02_features.ipynb

💡 TIP — Add this to every cell when loading data:
```python
# Pro tip: always load only the columns you need
# pd.read_parquet(path, columns=["timestamp","close"])
# This is 5-10x faster than loading all columns
```